In [1]:
import pandas as pd

df = pd.read_csv('bitcoin_dataset.csv', index_col=0)
print(f"Data shape: {df.shape}")
df.head()

Data shape: (1461, 6)


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2020-01-01,7200.174316,7200.174316,7254.330566,7174.944336,7194.892090,18565664997
2020-01-02,6985.470215,6985.470215,7212.155273,6935.270020,7202.551270,20802083465
2020-01-03,7344.884277,7344.884277,7413.715332,6914.996094,6984.428711,28111481032
2020-01-04,7410.656738,7410.656738,7427.385742,7309.514160,7345.375488,18444271275
2020-01-05,7411.317383,7411.317383,7544.497070,7400.535645,7410.451660,19725074095


In [2]:
df['benefit'] = df['Close'] - df['Open']
df['class'] = (df['benefit'] > 0).astype(int)

df['y'] = df['class'].shift(-1)
df['y_reg'] = df['Close'].shift(-1)

df = df.dropna()
df[['Open', 'Close', 'benefit', 'class', 'y', 'y_reg']].head()

,Open,Close,benefit,class,y,y_reg
Date,,,,,,
2020-01-01,7194.892090,7200.174316,5.282227,1,0.0,6985.470215
2020-01-02,7202.551270,6985.470215,-217.081055,0,1.0,7344.884277
2020-01-03,6984.428711,7344.884277,360.455566,1,1.0,7410.656738
2020-01-04,7345.375488,7410.656738,65.281250,1,1.0,7411.317383
2020-01-05,7410.451660,7411.317383,0.865723,1,1.0,7769.219238


In [3]:
from sklearn.model_selection import train_test_split

features = ['Open', 'High', 'Low', 'Close', 'Volume']
X = df[features]
y = df['y'].astype(int)
y_reg = df['y_reg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)
_, _, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42, shuffle=False)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

Train size: 1168 | Test size: 292


In [4]:
import time
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

start_time = time.time()
model_clf = XGBClassifier(random_state=42, eval_metric='logloss')
model_clf.fit(X_train, y_train)
time_classic_clf = time.time() - start_time

y_pred_clf = model_clf.predict(X_test)
acc_classic_clf = accuracy_score(y_test, y_pred_clf)

print(f"Classic Classification Time: {time_classic_clf:.4f} seconds")
print(f"Accuracy: {acc_classic_clf * 100:.2f}%\n")
print(classification_report(y_test, y_pred_clf))

Classic Classification Time: 0.5302 seconds
Accuracy: 52.74%

              precision    recall  f1-score   support

           0       0.55      0.32      0.41       146
           1       0.52      0.73      0.61       146

    accuracy                           0.53       292
   macro avg       0.53      0.53      0.51       292
weighted avg       0.53      0.53      0.51       292



In [5]:
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

start_time = time.time()
model_reg = XGBRegressor(random_state=42)
model_reg.fit(X_train, y_train_reg)
time_classic_reg = time.time() - start_time

y_pred_reg = model_reg.predict(X_test)
rmse_classic_reg = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

print(f"Classic Regression Time: {time_classic_reg:.4f} seconds")
print(f"RMSE: ${rmse_classic_reg:.2f}")

Classic Regression Time: 0.1063 seconds
RMSE: $1675.55
